In [0]:
from pyspark.sql.functions import upper, col , when , regexp_replace
from pyspark.sql.types import *

import os
import sys

project_path=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)

from util.Transformation import reuse


#Applying stream processing to load data from bronze to silver layer.


#Dimuser

In [0]:
df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimuser/checkpoint") \
    .load("abfss://metastore@meadls.dfs.core.windows.net/root/Bronze/dimuser")


In [0]:
df_user=df_user.withColumn("user_name",upper(col("user_name")))

df_user_obj=reuse()

df_user=df_user_obj.drop(df_user,"_rescued_data")
df_user=df_user.dropDuplicates(["user_id"])


In [0]:
df_user.writeStream.format("delta")\
    .option("checkpointLocation", "abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimuser/checkpoint")\
    .outputMode("append")\
    .trigger(once=True)\
    .option("path","abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimuser/data")\
    .toTable("spotify.silver.dimuser")

#Dimartist

In [0]:
df_art = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimart/checkpoint") \
    .load("abfss://metastore@meadls.dfs.core.windows.net/root/Bronze/dimartist")

In [0]:
df_art=df_art.withColumn("artist_name",upper(col("artist_name")))



In [0]:
df_art_obj=reuse()

df_art=df_art_obj.drop(df_art,"_rescued_data")
df_art=df_art.dropDuplicates(["artist_id"])

In [0]:
df_art.writeStream.format("delta")\
    .option("checkpointLocation", "abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimart/checkpoint")\
    .outputMode("append")\
    .trigger(once=True)\
    .option("path","abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimart/data")\
    .toTable("spotify.silver.dimart")


#Dimtrack

In [0]:
df_track = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimtrack/checkpoint") \
    .load("abfss://metastore@meadls.dfs.core.windows.net/root/Bronze/dimtrack")

In [0]:
df_track=df_track.withColumn("duration_flag",when((col("duration_sec")<200),"low")\
                                            .when((col("duration_sec")<300),"Medium")\
                                            .otherwise("high"))

df_track=df_track.withColumn("track_name",regexp_replace(col("track_name"),'-','='))

df_track_obj=reuse()
df_track=df_track_obj.drop(df_track,"_rescued_data")
#df_track=df_track.dropDuplicates(['track_id'])


In [0]:
df_track.writeStream.format("delta")\
    .option("checkpointLocation", "abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimtrack/checkpoint")\
    .outputMode("append")\
    .trigger(once=True)\
    .option("path","abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimtrack/data")\
    .toTable("spotify.silver.dimtrack")

#Dimdate

In [0]:
df_date=spark.readStream.format("cloudFiles")\
                        .option("cloudFiles.format","parquet")\
                        .option("cloudFiles.schemaLocation","abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimdate/checkpoint")\
                        .load("abfss://metastore@meadls.dfs.core.windows.net/root/Bronze/dimdate")

In [0]:
df_date=df_date.drop("df_date","_rescued_data")

df_date.writeStream.format("delta")\
    .option("checkpointLocation", "abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimdate/checkpoint")\
    .outputMode("append")\
    .trigger(once=True)\
    .option("path","abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimdate/data")\
    .toTable("spotify.silver.dimdate")



#Factstream

In [0]:
df_fact=spark.readStream.format("cloudFiles")\
                        .option("cloudFiles.format","parquet")\
                        .option("cloudFiles.schemaLocation","abfss://metastore@meadls.dfs.core.windows.net/root/Silver/factstream/checkpoint")\
                        .load("abfss://metastore@meadls.dfs.core.windows.net/root/Bronze/factstream")

In [0]:
df_fact=reuse().drop(df_fact,"_rescued_data")

df_fact.writeStream.format("delta")\
    .option("checkpointLocation", "abfss://metastore@meadls.dfs.core.windows.net/root/Silver/dimfact/checkpoint")\
    .outputMode("append")\
    .trigger(once=True)\
    .option("path","abfss://metastore@meadls.dfs.core.windows.net/root/Silver/factstream/data")\
    .toTable("spotify.silver.factstream")

In [0]:
%sql
select * from spotify.silver.dimtrack where track_id in (46,5);

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,_rescued_data,duration_flag
46,Expanded foreground knowledgebase,225,Democrat Album,254,2023-09-08,2025-09-14T19:49:55.000Z,null,Medium
5,Multi=layered needs=based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,null,low
